# AutoGen Lesson 2: Tools (Local Ollama + `gemma4:e2b`)

This notebook teaches how to add and use Python tools with AutoGen locally.

Learning path:
1. Setup and checks
2. Tool-ready model client
3. Define Python tools
4. First tool call
5. Multi-tool task
6. Mini challenge


## 0) Install (Run Once)
Run this only if your current kernel is missing dependencies.

In [ ]:
# %pip install -U autogen-agentchat "autogen-ext[ollama]"

## 1) Environment Check

In [ ]:
import importlib
import platform
import subprocess

print(f"Python: {platform.python_version()}")
for pkg in ["autogen_agentchat", "autogen_ext"]:
    module = importlib.import_module(pkg)
    print(f"{pkg}: {getattr(module, '__version__', 'version not exposed')}")

model_tag = "gemma4:e2b"
models = subprocess.check_output(["ollama", "list"], text=True)
print("\nInstalled Ollama models:")
print(models)
if model_tag not in models:
    raise RuntimeError(f"Model {model_tag} not found. Run: ollama pull {model_tag}")

print(f"Model check passed: {model_tag}")


## 2) Build a Tool-Ready Client
For tool-calling, we set `function_calling=True` in `model_info`.

Note: `gemma4:e2b` is slower than smaller models, so we keep responses short with `num_predict`.

In [ ]:
from autogen_ext.models.ollama import OllamaChatCompletionClient

MODEL_TAG = "gemma4:e2b"
MODEL_INFO_TOOLS = {
    "vision": False,
    "function_calling": True,
    "json_output": False,
    "family": "unknown",
    "structured_output": False,
}

def build_tool_client() -> OllamaChatCompletionClient:
    return OllamaChatCompletionClient(
        model=MODEL_TAG,
        model_info=MODEL_INFO_TOOLS,
        host="http://localhost:11434",
        options={"temperature": 0, "num_predict": 96},
    )

print("Tool-ready client helper created.")


## 3) Define Tools
These are plain Python functions. AutoGen exposes them to the model as callable tools.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b

def word_count(text: str) -> int:
    """Count words in a text string."""
    return len(text.split())

def current_time_ist() -> str:
    """Return current time in Asia/Kolkata timezone."""
    now = datetime.now(ZoneInfo("Asia/Kolkata"))
    return now.strftime("%Y-%m-%d %H:%M:%S %Z")

TOOLS = [multiply, word_count, current_time_ist]
print([t.__name__ for t in TOOLS])


## 4) First Tool Call (Arithmetic)
This run returns tool events so you can see exactly what happened.

In [ ]:
from autogen_agentchat.agents import AssistantAgent

def print_messages(task_result) -> None:
    for i, msg in enumerate(task_result.messages):
        print(f"\n[{i}] {type(msg).__name__} | source={getattr(msg, 'source', None)}")
        print(getattr(msg, "content", ""))

client = build_tool_client()
agent = AssistantAgent(
    name="tool_agent",
    model_client=client,
    tools=[multiply],
    system_message="For arithmetic, always call the multiply tool.",
    max_tool_iterations=1,
    reflect_on_tool_use=False,
)

result = await agent.run(task="Use tool: multiply 37 and 19")
print_messages(result)

await client.close()


## 5) Multi-Tool Task
Now the agent can choose from multiple tools.

In [ ]:
client = build_tool_client()
agent = AssistantAgent(
    name="multi_tool_agent",
    model_client=client,
    tools=TOOLS,
    system_message=(
        "Use the available tools whenever useful. "
        "If a tool gives the answer, return it directly and briefly."
    ),
    max_tool_iterations=2,
    reflect_on_tool_use=False,
)

task = (
    "1) Count words in this sentence: AutoGen tools make local agents practical and reliable. "
    "2) Multiply 12 and 11. "
    "3) Show current IST time."
)

result = await agent.run(task=task)
print_messages(result)

await client.close()


## 6) Optional: Final Natural-Language Answer After Tools
Set `reflect_on_tool_use=True` to let the model write a final polished reply.
This may be slower for larger local models.

In [ ]:
client = build_tool_client()
agent = AssistantAgent(
    name="reflect_agent",
    model_client=client,
    tools=[multiply],
    system_message="Use tool first, then explain in one sentence.",
    max_tool_iterations=1,
    reflect_on_tool_use=True,
)

result = await agent.run(task="What is 14 * 15?")
print_messages(result)

await client.close()


## 7) Mini Challenge
Create your own tool below and add it to `TOOLS`.

Ideas:
- `c_to_f(celsius: float) -> float`
- `estimate_read_time(words: int) -> str`
- `slugify(text: str) -> str`